1. Install Dependencies

In [ ]:
pip install transformers torch wikipedia

2. Load a Pre‑trained Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import wikipedia

# Load model and tokenizer
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

3. Create the Chat Loop

In [ ]:
# Chat history
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

while True:
    user_input = input("User: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # Try Wikipedia first
    try:
        wiki_result = wikipedia.summary(user_input, sentences=2)
        print("Chatbot:", wiki_result)
        continue
    except:
        pass

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append to chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95
    )

    # Decode response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)